# Stochastic Calculus, Step by Step · Lesson 10: The multi-period binomial model

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **Part 2's teaching tree.** A share at S₀ = 100 that, every four months, goes up 10% (heads, H) or down 10% (tails, T); three steps make the course's one year. The real-world chance of heads is p = 0.6. The risk-free rate is the course's r = 3% (continuously compounded), so cash grows by R = e^0.01 per step, and the strike is K = 105. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. Working backward through three steps · 3. One formula for the whole tree · 4. Hedging along a path · 5. Put-call parity · 6. Many steps: the tree meets Black-Scholes · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 10,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
import itertools

S0_, u, d, p = 100.0, 1.1, 0.9, 0.6          # the teaching tree: +10% (H) or -10% (T) per step, P(H) = 0.6
steps, dt_ = 3, 1 / 3                         # three steps of four months
R = math.exp(r * dt_)                         # cash growth per step
worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]   # HHH, HHT, ..., TTT


def price(w, n):
    """The share price after the first n tosses of world w."""
    return S0_ * u ** w[:n].count("H") * d ** w[:n].count("T")


def prob(w, q=p):
    """The probability of world w (or of the first tosses of one) when heads has probability q."""
    return q ** w.count("H") * (1 - q) ** w.count("T")


out["tree"] = {"S0": S0_, "u": u, "d": d, "p": p, "steps": steps, "dt": dt_, "R": R, "K": K}
pd.DataFrame({"world": worlds, "S1": [price(w, 1) for w in worlds], "S2": [price(w, 2) for w in worlds],
              "S3": [price(w, 3) for w in worlds], "P": [prob(w) for w in worlds]}).round(4)

## 2. Working backward through three steps

The call with strike 105 now expires after all three steps. At expiry its value is known at each node. One step earlier, each node is a one-period problem (Lesson 9): copy the two values that can follow it with $\Delta$ shares and cash $B$, or equivalently take the discounted risk-neutral average. Repeat back to today.

On a recombining tree a node is fixed by the step $n$ and the number of heads $k$, with price $S_0u^kd^{\,n-k}$.

In [ ]:
pt = (R - d) / (u - d)
S = lambda n, k_: S0_ * u**k_ * d ** (n - k_)


def backward(payoff, n_steps=steps):
    # Values V[(n, k)] and the copy (Delta, B) at each node before expiry.
    V = {(n_steps, k_): payoff(S(n_steps, k_)) for k_ in range(n_steps + 1)}
    hedge = {}
    for n in range(n_steps - 1, -1, -1):
        for k_ in range(n + 1):
            Vu_, Vd_ = V[(n + 1, k_ + 1)], V[(n + 1, k_)]
            V[(n, k_)] = (pt * Vu_ + (1 - pt) * Vd_) / R
            D_ = (Vu_ - Vd_) / (S(n + 1, k_ + 1) - S(n + 1, k_))
            hedge[(n, k_)] = {"Delta": D_, "B": V[(n, k_)] - D_ * S(n, k_)}
    return V, hedge


call = lambda s: max(s - K, 0.0)
V, hedge = backward(call)
nodes = [{"t": n, "k": k_, "price": S(n, k_), "value": V[(n, k_)],
          "Delta": hedge[(n, k_)]["Delta"] if n < steps else None, "B": hedge[(n, k_)]["B"] if n < steps else None}
         for n in range(steps + 1) for k_ in range(n + 1)]
out["back"] = {"pt": pt, "V0": V[(0, 0)], "nodes": nodes, "Delta0": hedge[(0, 0)]["Delta"], "B0": hedge[(0, 0)]["B"]}
pd.DataFrame(nodes).round(4)

## 3. One formula for the whole tree

Unrolling the backward steps: a path with $k$ heads has risk-neutral probability $\tilde p^k(1 - \tilde p)^{N - k}$, and there are $\binom Nk$ such paths, all ending at the same price. So

$$V_0 = \frac{1}{R^N}\sum_{k=0}^N\binom Nk\tilde p^{\,k}(1 - \tilde p)^{N - k}\,\max\big(S_0u^kd^{\,N - k} - K, 0\big) .$$

In [ ]:
terms = [{"k": k_, "paths": math.comb(steps, k_), "q": math.comb(steps, k_) * pt**k_ * (1 - pt) ** (steps - k_),
          "S3": S(steps, k_), "payoff": call(S(steps, k_))} for k_ in range(steps, -1, -1)]
one_shot = sum(t_["q"] * t_["payoff"] for t_ in terms) / R**steps
assert abs(one_shot - V[(0, 0)]) < 1e-12
out["oneshot"] = {"terms": terms, "V0": one_shot, "disc": R ** -steps}
pd.DataFrame(terms).round(4)

## 4. Hedging along a path

Sell the call for $V_0$ and hedge: hold $\Delta$ shares and put the rest in cash; after each toss, move to the new node's $\Delta$, paying for the extra shares from the cash account. Nothing is added or taken out (the hedge is **self-financing**), yet at expiry the account equals the call's payoff, in every world. Path HTH, step by step:

In [ ]:
def hedge_path(w):
    rows, cash, k_ = [], V[(0, 0)] - hedge[(0, 0)]["Delta"] * S0_, 0
    shares = hedge[(0, 0)]["Delta"]
    rows.append({"step": 0, "toss": "", "S": S0_, "Delta": shares, "cash": cash, "portfolio": V[(0, 0)], "option": V[(0, 0)]})
    for n, toss in enumerate(w, start=1):
        cash *= R
        k_ += toss == "H"
        port = shares * S(n, k_) + cash                     # value before rebalancing
        new = hedge[(n, k_)]["Delta"] if n < steps else 0.0   # at expiry, sell everything
        cash -= (new - shares) * S(n, k_)                   # buy or sell shares out of the cash account
        shares = new
        rows.append({"step": n, "toss": toss, "S": S(n, k_), "Delta": shares, "cash": cash,
                     "portfolio": port, "option": V[(n, k_)]})
    return rows


path_rows = hedge_path("HTH")
errors = {w: hedge_path(w)[-1]["portfolio"] - call(price(w, steps)) for w in worlds}
assert max(abs(e) for e in errors.values()) < 1e-9
out["hedge"] = {"rows": path_rows, "max_error": max(abs(e) for e in errors.values()), "world": "HTH"}
pd.DataFrame(path_rows).round(4)

## 5. Put-call parity

Owning a call and selling a put with the same strike and expiry pays $S_3 - K$ in every world, so it must cost what that costs: one share minus the discounted strike. **$C - P = S_0 - K/R^N$**, whatever the model.

In [ ]:
Vp, hedge_p = backward(lambda s: max(K - s, 0.0))
out["parity"] = {"C": V[(0, 0)], "P": Vp[(0, 0)], "lhs": V[(0, 0)] - Vp[(0, 0)], "rhs": S0_ - K / R**steps,
                 "K_disc": K / R**steps, "Delta_put0": hedge_p[(0, 0)]["Delta"]}
assert abs(out["parity"]["lhs"] - out["parity"]["rhs"]) < 1e-12
out["parity"]

## 6. Many steps: the tree meets Black-Scholes

Banks build trees with many short steps and size the moves from the volatility: $u = e^{\sigma\sqrt{\Delta t}}$, $d = 1/u$ (the Cox-Ross-Rubinstein tree). With the course's numbers ($\sigma = 20\%$, $T = 1$) the tree price settles on the Black-Scholes value of Part 4, swinging around it as $N$ grows.

In [ ]:
def crr(N, payoff=call, american=False):
    dt = T / N
    uu = math.exp(sigma * math.sqrt(dt)); dd = 1 / uu; RR = math.exp(r * dt)
    q = (RR - dd) / (uu - dd)
    s = S0 * uu ** np.arange(N, -1, -1) * dd ** np.arange(0, N + 1)
    v = np.array([payoff(x) for x in s])
    for n in range(N - 1, -1, -1):
        s = s[:-1] / uu
        v = (q * v[:-1] + (1 - q) * v[1:]) / RR
        if american:
            v = np.maximum(v, [payoff(x) for x in s])
    return float(v[0])


d1 = (math.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T)); d2 = d1 - sigma * math.sqrt(T)
bs = S0 * N(d1) - K * math.exp(-r * T) * N(d2)
conv = [{"N": n, "price": crr(n), "error": crr(n) - bs} for n in (3, 10, 30, 100, 300, 1000)]
curve = list(range(2, 201))
out["crr"] = {"bs": bs, "rows": conv, "N": curve, "prices": [crr(n) for n in curve],
              "u3": math.exp(sigma * math.sqrt(T / 3)), "d3": math.exp(-sigma * math.sqrt(T / 3))}
out["crr"]["bs_line"] = [bs] * len(curve)
out["crr"]["ticks"] = [{"x": x, "label": str(x)} for x in (0, 50, 100, 150, 200)]
pd.DataFrame(conv)

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 2: Solutions*.

**Exercise 1.** Price a call with strike 105 that expires after **two** steps (eight months), by backward induction on the teaching tree.

**Exercise 2.** For the two-step call of Exercise 10.1, find $\Delta$ and the cash $B$ at each of the three nodes before expiry. Why is $\Delta$ 0 at the lower node after one step?

**Exercise 3.** Price the three-step put with strike 105 by backward induction and check put-call parity $C - P = S_0 - K/R^3$ with the call price from the lesson.

**Exercise 4.** Write the one-shot formula for $N = 2$ and check that it gives your answer to Exercise 10.1.

**Exercise 5.** In the hedge of the three-step call, show that rebalancing after the first toss (heads) costs nothing: the portfolio's value just before rebalancing equals the cost of the new portfolio.

**Exercise 6.** A forward contract pays $S_3 - 105$ at the end. Show that its value at step $n$ is $S_n - 105/R^{3 - n}$ and that its $\Delta$ is 1 at every node. Why does a forward need no rebalancing?

**Exercise 7.** (Code) Write a CRR pricer for $N$ steps and tabulate the call price for $N = 50, 51, 52, \dots, 60$. Describe how the error behaves for even and odd $N$.

**Exercise 8.** (Code) Run the three-step hedge along all eight worlds and print the final error. Then repeat with a lazy hedge that keeps the first $\Delta$ to the end. How large do its errors get?

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")